# Week 10 - Document Classification (1/2)
**Notebook 1 : Naïve Bayes and k-Nearest Neighbor**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

Weeks 4 to 7 turned a document into a row of numbers. Lecture 9-2 *Document Classification* finally uses that row:
a classifier reads it and returns a class. This notebook builds the lecture's two classifiers by hand, checks them
against scikit-learn, and runs them on 5,000 news articles.

| Lecture chapter | What we do in code |
|---|---|
| 1. Overview | the four news categories, and a document as a row of counts |
| 2. Naïve Bayesian Classifier | the lecture's two reviews by hand, MLE from counts, the zero-probability problem, Laplace smoothing |
| 3. k-Nearest Neighbor | Minkowski distance, majority vs weighted voting, $k$ too small or too large, choosing $k$ on validation data, normalization, cut-off |
| (both) | Naïve Bayes vs k-NN on the same 1,000 test articles, and on a headline you write yourself |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom with
> `Shift + Enter`. Run the **Setup** cells first. The **🤔 Quick check** notes after an output say what to look for in it.
> Two cells marked **🔧** ask you to change one value in the code and run the cell again.

## &nbsp;0. Setup
NumPy, pandas, scikit-learn and Matplotlib. All pre-installed in Colab, nothing to install.

**The data: AG News.** News articles in four categories (World, Sports, Business, Sci/Tech). The full corpus has
120,000 training and 7,600 test articles; we use a balanced sample of **1,000 training and 250 test articles per category**,
so that every cell runs in a few seconds. `text` is the headline followed by the first sentences of the article.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing import normalize

%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 80)
np.set_printoptions(linewidth=140, suppress=True)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week10-document-classification/data/ag_news_sample.csv"

news = pd.read_csv(DATA_URL)
train = news[news['split'] == 'train'].reset_index(drop=True)
test = news[news['split'] == 'test'].reset_index(drop=True)

CLASSES = ['World', 'Sports', 'Business', 'Sci/Tech']
y_train = train['category'].map(CLASSES.index).to_numpy()      # 0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech
y_test = test['category'].map(CLASSES.index).to_numpy()

print('train :', len(train), ' test :', len(test))
print(train['category'].value_counts().to_dict())

## &nbsp;1. Document classification is supervised learning
*Document Classification: Definition* (lecture p. 8)

* **Input** : a document $d$ and a fixed set of classes $C = \{c_1, c_2, \dots, c_J\}$
* **Training set** : $m$ hand-labeled documents $(d_1, c_1), \dots, (d_m, c_m)$
* **Output** : a learned classifier $\gamma : d \to c$

Here $J = 4$ and $m = 4{,}000$. One headline per class:

In [ ]:
train.groupby('category', observed=True).head(1)[['category', 'title']]

*Supervised Machine Learning with Bag-of-Words representation* (p. 9-10): the classifier never reads the sentence.
It reads a row of counts, exactly the term-document matrix of Week 4. The vocabulary is learned from the **training**
articles only, and the test articles are counted with that same vocabulary.

In [ ]:
vec = CountVectorizer(stop_words='english', min_df=2)     # words in at least 2 training articles
X_train = vec.fit_transform(train['text'])                 # learn the vocabulary, count
X_test = vec.transform(test['text'])                       # count with the same vocabulary
vocab = vec.get_feature_names_out()

print('X_train :', X_train.shape, '  X_test :', X_test.shape)

row = X_train[0]
print('\n' + train.loc[0, 'title'], ' ->', train.loc[0, 'category'])
pd.Series(row.data, index=vocab[row.indices]).sort_values(ascending=False).head(8).to_frame('count').T

**🤔 Quick check** This is the table on p. 10: a few words with their counts, and everything else zero. Out of 7,982
columns, an article fills about 19 on average (this short one only 8). A classifier is a function from this row to one of the four classes.

## &nbsp;2. Naïve Bayesian Classifier
*Bayes' Rule* and the *naïve* assumption (p. 15): the words are treated as independent of each other given the class, so

$$P(c \mid w_1, \dots, w_n) \;\propto\; P(c)\,\prod_i P(w_i \mid c)$$

The classifier picks the class with the largest right-hand side. Two kinds of numbers are needed: a **prior** $P(c)$ per
class and a **likelihood** $P(w \mid c)$ per word and class.

### &nbsp;2-1. The lecture's two reviews, by hand
*Example: Classifying movie reviews into Positive/Negative class* (p. 17-19). The slide gives $P(w \mid c)$ directly.
Both classes are taken as equally likely, so only the products of the likelihoods are compared.

In [ ]:
P_word = pd.DataFrame({                    # the table of p. 17
    'Pos': [0.1, 0.1, 0.1, 0.4, 0.2, 0.3, 0.5, 0.1, 0.02, 0.1, 0.02, 0.02, 0.15],
    'Neg': [0.1, 0.1, 0.1, 0.01, 0.2, 0.05, 0.05, 0.1, 0.3, 0.1, 0.35, 0.02, 0.15],
}, index=['this', 'movie', 'was', 'awesome', 'i', 'really', 'enjoyed', 'it',
          'boring', 'and', 'waste', 'of', 'time'])

reviews = {'Review 1': 'this movie was awesome i really enjoyed it',
           'Review 2': 'this movie was boring and waste of time'}

for name, text in reviews.items():
    words = text.split()
    pos = P_word.loc[words, 'Pos'].prod()          # product of P(word | Pos)
    neg = P_word.loc[words, 'Neg'].prod()          # product of P(word | Neg)
    print(f'{name}: Pos {pos / 1e-8:8.3f} x 1e-8   Neg {neg / 1e-8:8.3f} x 1e-8   ->',
          'Positive' if pos > neg else 'Negative')

**🤔 Quick check** Review 1: $120 \times 10^{-8}$ against $0.05 \times 10^{-8}$, Positive. Review 2: $0.012$ against $3.15$,
Negative. The slide on p. 18 prints $0.5 \times 10^{-8}$ for the negative side of Review 1, but the product of its own
table is $0.05 \times 10^{-8}$ (one zero is missing on the slide). The decision is the same either way.

Notice which words decide: *this, movie, was, it* have the same probability in both columns and cancel out.
*awesome* and *enjoyed* carry Review 1, *boring* and *waste* carry Review 2.

### &nbsp;2-2. Where the probabilities come from : maximum likelihood estimates
*Maximum likelihood estimates* (p. 16): *simply use the frequencies in the data*.

$$\hat P(c_j) = \frac{N.Doc(C = c_j)}{\text{Total number of documents}} \qquad
\hat P(w_i \mid c_j) = \frac{count(w_i, c_j)}{\sum_{w \in V} count(w, c_j)}$$

$count(w, c)$ is the number of times $w$ appears in all training articles of class $c$. One matrix holds all of them:
one row per class, one column per word.

In [ ]:
counts = np.vstack([np.asarray(X_train[y_train == c].sum(axis=0)).ravel()      # count(w, c) for every word
                    for c in range(len(CLASSES))])
prior = np.array([(y_train == c).mean() for c in range(len(CLASSES))])           # P(c)
P_mle = counts / counts.sum(axis=1, keepdims=True)                               # P(w | c), each row sums to 1

print('counts :', counts.shape, '  (classes x words)')
print('prior  :', prior)
print('total words per class :', counts.sum(axis=1))

words = ['iraq', 'season', 'oil', 'shares', 'software', 'game']
cols = [vec.vocabulary_[w] for w in words]
pd.concat({'count(w, c)': pd.DataFrame(counts[:, cols], index=CLASSES, columns=words),
           'P(w | c)': pd.DataFrame(P_mle[:, cols], index=CLASSES, columns=words).round(5)})

**🤔 Quick check** Each word leans clearly toward one category: *iraq* to World, *season* and *game* to Sports, *oil* and
*shares* to Business, *software* to Sci/Tech. The prior is 0.25 for every class because the sample is balanced.
And *iraq* never occurs in a Sports article: $\hat P(\text{iraq} \mid \text{Sports}) = 0$.

### &nbsp;2-3. The zero-probability problem
*Limitation* (p. 16): *what if we have seen no training document with the word "fantastic" and classified in the class
"positive"?* Then $\hat P(\text{fantastic} \mid \text{positive}) = 0$, and the whole product is 0,
*no matter the other evidence*. On our test articles:

In [ ]:
def mle_products(i):
    # P(c) * prod_i P(w_i | c) for test article i, for every class (a word that occurs twice counts twice)
    x = X_test[i]
    return prior * np.prod(P_mle[:, x.indices] ** x.data, axis=1)


products = np.array([mle_products(i) for i in range(X_test.shape[0])])
print('test articles                                  :', len(products))
print('product = 0 for the true class                 :', (products[np.arange(len(y_test)), y_test] == 0).sum())
print('product = 0 for ALL four classes (no decision) :', (products == 0).all(axis=1).sum())

One of those articles, and for every class the words that make its product zero:

In [ ]:
i = test.index[test['title'] == 'Foreign reactions to the election'][0]
x = X_test[i]
print(test.loc[i, 'text'], '\n  ->', test.loc[i, 'category'])
print('\nproducts :', mle_products(i))
for c, name in enumerate(CLASSES):
    unseen = [vocab[j] for j in x.indices if counts[c, j] == 0]
    print(f'{name:<9} words never seen in this class : {unseen}')

**🤔 Quick check** More than half of the test articles get 0 for **every** class: the classifier cannot decide at all.
This World article is lost because of one word, *accepting*, that no World training article happens to contain.

### &nbsp;2-4. Laplace (add-1) smoothing
*Smoothing Techniques* (p. 20): add 1 to every count, so that no probability is ever zero.

$$\hat P(w_i \mid c_j) = \frac{count(w_i, c_j) + 1}{\sum_{w \in V} \big(count(w, c_j) + 1\big)}
= \frac{count(w_i, c_j) + 1}{\sum_{w \in V} count(w, c_j) + |V|}$$

A practical detail: a product of 30 numbers around $10^{-4}$ is around $10^{-120}$. Computers handle that better as a
**sum of logarithms**, and since $\log$ is increasing, the class with the largest sum is the class with the largest product.
For a whole matrix of counts, the sum over the words of $count \times \log P(w \mid c)$ is one matrix product.

In [ ]:
V = len(vocab)
P_laplace = (counts + 1) / (counts.sum(axis=1, keepdims=True) + V)        # p. 20

log_scores = X_test @ np.log(P_laplace).T + np.log(prior)                 # log P(c) + sum count * log P(w | c)
pred_hand = log_scores.argmax(axis=1)

print('|V| =', V)
print('smallest P(w | c) now :', P_laplace.min())
print('accuracy (by hand)    :', accuracy_score(y_test, pred_hand))

### &nbsp;2-5. The same classifier in scikit-learn
`MultinomialNB` is exactly this model: priors from the class shares, likelihoods from the word counts, and `alpha` is the
number added to every count (`alpha=1.0` is Laplace smoothing). It stores $\log \hat P(w \mid c)$ in `feature_log_prob_`.

In [ ]:
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

print('accuracy (scikit-learn)          :', accuracy_score(y_test, pred_nb))
print('same predictions as by hand      :', (pred_nb == pred_hand).all())
print('same log P(w | c) as by hand     :', np.allclose(nb.feature_log_prob_, np.log(P_laplace)))

How much should be added? The same model with six values of `alpha`, from almost nothing to 2:

In [ ]:
rows = []
for a in [1e-10, 0.01, 0.1, 0.5, 1.0, 2.0]:
    acc = MultinomialNB(alpha=a).fit(X_train, y_train).score(X_test, y_test)
    rows.append({'alpha': a, 'test accuracy': acc})
pd.DataFrame(rows).set_index('alpha').T

**🤔 Quick check** By hand and scikit-learn agree on all 1,000 predictions. `alpha` close to 0 is the unsmoothed MLE
of 2-2: scikit-learn keeps a tiny positive value, so it still decides, but it is the worst setting in the table.
Any reasonable amount of smoothing lands around 0.87.

## &nbsp;3. k-Nearest Neighbor
*"Birds of a feather flock together"* (p. 22): a new document gets the class of the training documents closest to it.
There is no model to estimate: the training data **is** the model. The lecture's *k-NN Classification Process* has five
steps. We follow them in order, except that Step 4 (the voting rule) comes right after the first neighbors, because it
answers the tie we meet there.

### &nbsp;3-1. Step 1 : prepare the reference data
*Define attributes: BoW representation or distributed representation* (p. 26). We use the TF-IDF matrix of Week 4,
with the same vocabulary settings. Every row is divided by its length (`norm='l2'`, the default), which matters in 3-7.

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', min_df=2)
T_train = tfidf.fit_transform(train['text'])
T_test = tfidf.transform(test['text'])
print('T_train :', T_train.shape, '  T_test :', T_test.shape)
print('length of every row :', np.unique(np.round(np.sqrt(T_train.multiply(T_train).sum(axis=1)), 6)))

### &nbsp;3-2. Step 2 : define the similarity measure
*Similarity* $\propto$ *1 / distance*, with the *Minkowski distance with order p* (p. 27):

$$distance(P, Q) = \Big(\sum_{i=1}^{n} \lvert x_i - y_i \rvert^{p}\Big)^{1/p}
\qquad p = 2 : \text{Euclidean}, \quad p = 1 : \text{Manhattan}$$

First on the two-attribute table of p. 26, with a new point $x = (2.0, 0.5)$:

In [ ]:
def minkowski(P, Q, p):
    return (np.abs(P - Q) ** p).sum() ** (1 / p)


ref = pd.DataFrame({'Vision': [2.54, 1.78, 0.12, 0.25, 3.12], 'Finance': [0.15, 0.00, 2.65, 3.52, 0.14],
                    'Class': ['TPAMI', 'TPAMI', 'JoF', 'JoF', 'TPAMI']}, index=['Doc 1', 'Doc 2', 'Doc 3', 'Doc 4', 'Doc N'])
x_new = np.array([2.0, 0.5])

ref['p = 1 (Manhattan)'] = [minkowski(x_new, r, 1) for r in ref[['Vision', 'Finance']].to_numpy()]
ref['p = 2 (Euclidean)'] = [minkowski(x_new, r, 2) for r in ref[['Vision', 'Finance']].to_numpy()]
ref.round(3)

Now the same Euclidean distance between one test article and **all** 4,000 training articles, and its 5 nearest neighbors.

In [ ]:
q = 0
dist = euclidean_distances(T_test[q], T_train).ravel()     # one distance per training article
nearest = np.argsort(dist)[:5]

print('query :', test.loc[q, 'title'], ' ->', test.loc[q, 'category'], '\n')
neighbors = train.loc[nearest, ['title', 'category']].assign(distance=dist[nearest].round(3))
neighbors

**🤔 Quick check** All five neighbors share the word *Argentina*, and that is the whole story: k-NN only knows which
articles look alike. Two neighbors are Business, two are Sports, one is Sci/Tech. With $k = 5$ the vote is a tie, which is
where the combining rule comes in.

### &nbsp;3-3. Step 4 : determine the combining rule
*Majority voting vs. Weighted voting* (p. 30). Weighted voting gives each neighbor the weight $1/distance$, normalized to
sum to 1. The lecture's table first:

In [ ]:
lab = np.array(['TPAMI', 'JoF', 'TPAMI', 'JoF', 'JoF'])     # neighbors N1 ... N5 of p. 30
d = np.array([1, 2, 3, 4, 5])

w = (1 / d) / (1 / d).sum()
print('weights :', w.round(2))
print('majority voting : P(X in TPAMI) =', np.mean(lab == 'TPAMI'))
print('weighted voting : P(X in TPAMI) =', round(w[lab == 'TPAMI'].sum(), 3))

**🤔 Quick check** 0.4 against 0.584: with a cut-off of 0.5, majority voting says JoF and weighted voting says TPAMI, as on
the slide. The slide prints 0.59 and a last weight of 0.08: it adds the **rounded** weights 0.44 + 0.15. Unrounded, the
weights are 0.438, 0.219, 0.146, 0.109, 0.088 and the sum for TPAMI is 0.584. Now the tied vote of 3-2:

In [ ]:
def vote(labels, distances, weighted):
    w = 1 / distances if weighted else np.ones(len(labels))
    w = w / w.sum()
    return pd.Series(w, index=labels).groupby(level=0).sum().sort_values(ascending=False).round(3)


print('majority :', vote(neighbors['category'].to_numpy(), neighbors['distance'].to_numpy(), False).to_dict())
print('weighted :', vote(neighbors['category'].to_numpy(), neighbors['distance'].to_numpy(), True).to_dict())

**🤔 Quick check** Majority voting has no answer (0.4 / 0.4 / 0.2). Weighted voting breaks the tie for Business, by less than 0.01,
because the closest neighbor is a Business article. (The true class is World: no rule can fix neighbors that only share
a country name.)

### &nbsp;3-4. Step 3 : candidate values for $k$, seen on 2-D points
*If k is too small, then the classification will be highly locally sensitive (over-fitting). If k is too large, then it will
lose the ability to capture the local structure (under-fitting)* (p. 28).

News articles have 7,982 dimensions, so we cannot look at them. 300 points in two dimensions (two interleaving half-moons,
two classes) we can: 200 points to learn from and 100 to test on. The first cell only makes the points.

In [ ]:
from sklearn.datasets import make_moons

P2, c2 = make_moons(n_samples=300, noise=0.35, random_state=42)          # 2-D points, two classes
P2_tr, P2_te, c2_tr, c2_te = train_test_split(P2, c2, test_size=100, random_state=42, stratify=c2)
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.6, 2.1, 300))   # a grid that covers the plot
print('training points :', P2_tr.shape, '  test points :', P2_te.shape)

The next cell colors every point of the plane by the class k-NN would give it. The border between the two colors is the
**decision boundary**. It starts with $k = 1$.

> 🔧 **Change and run again.** After the first run, change `k = 1` to `k = 15` and run the cell again, then to `k = 151`.
> Watch how the shape of the boundary changes.

In [ ]:
k = 1                                    # 🔧 change to 15, then to 151, and run this cell again

knn2 = KNeighborsClassifier(n_neighbors=k).fit(P2_tr, c2_tr)
zz = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)       # the class of every grid point

fig, ax = plt.subplots(figsize=(7, 4.3))
ax.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
ax.scatter(P2_tr[:, 0], P2_tr[:, 1], c=c2_tr, cmap='coolwarm', s=14, edgecolor='k', linewidth=0.3)
ax.set_title(f'k = {k}    training accuracy {knn2.score(P2_tr, c2_tr):.2f}    test accuracy {knn2.score(P2_te, c2_te):.2f}')
ax.set_xticks([]); ax.set_yticks([])
plt.show()

**🤔 Quick check** With $k = 1$ the boundary draws a small island around every stray point: training accuracy 1.00, test
accuracy 0.83. That is **over-fitting**. With $k = 15$ there is one smooth boundary between the two moons, and the test
accuracy rises to 0.87. With $k = 151$ (three quarters of the training points vote every time) the boundary is almost a
straight line, and both accuracies fall to about 0.78. That is **under-fitting**. The lecture's sentence on p. 28, in
three pictures.

### &nbsp;3-5. Step 5 : the best $k$ on validation data
*Use the validation data* (p. 31). Choosing $k$ by eye only works in two dimensions. For the news articles we hold out a
quarter of the training articles as validation data, try every odd $k$ from 1 to 51, and keep the $k$ with the smallest
validation error.

In [ ]:
T_tr, T_val, y_tr, y_val = train_test_split(T_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

ks = list(range(1, 52, 2))
train_err, val_err = [], []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(T_tr, y_tr)
    train_err.append(1 - knn.score(T_tr, y_tr))          # error on the data it was given
    val_err.append(1 - knn.score(T_val, y_val))          # error on data it has never seen

best_k = ks[int(np.argmin(val_err))]
err = pd.DataFrame({'% Error Training': np.round(100 * np.array(train_err), 2),
                    '% Error Validation': np.round(100 * np.array(val_err), 2)}, index=pd.Index(ks, name='k'))
print('best k :', best_k)
err.loc[[1, 3, 5, 9, 15, best_k, 25, 31, 41, 51]]

The same two columns as a picture, with the best $k$ marked by the dashed line.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(ks, err['% Error Training'], 'o-', label='training')
ax.plot(ks, err['% Error Validation'], 's-', label='validation')
ax.axvline(best_k, color='gray', ls='--', lw=1)
ax.set_xlabel('k'); ax.set_ylabel('% error'); ax.legend(); ax.set_title(f'k-NN on TF-IDF : best k = {best_k}')
plt.show()

**🤔 Quick check** $k = 1$ has **0 %** training error, as in the first row of the slide's table on p. 31: every training
article is its own nearest neighbor. That number says nothing about new articles; its validation error is the worst of
all. The validation error falls quickly and then stays nearly flat: from about $k = 15$ on, the differences are a few
articles out of 1,000, so the exact best $k$ would move with another validation split.

### &nbsp;3-6. The same classifier in scikit-learn
`KNeighborsClassifier` covers every step: `n_neighbors` is $k$, `p` is the Minkowski order (default 2),
`weights='uniform'` is majority voting and `weights='distance'` is weighted voting.

In [ ]:
rows = []
for weights in ['uniform', 'distance']:
    for p in [2, 1]:
        knn = KNeighborsClassifier(n_neighbors=best_k, weights=weights, p=p)
        knn.fit(T_train, y_train)
        rows.append({'weights': weights, 'p': p, 'test accuracy': knn.score(T_test, y_test)})
pd.DataFrame(rows)

**🤔 Quick check** The two voting rules hardly differ here. The distance does: Manhattan ($p = 1$) collapses below 0.3.
The reason is the same as in the next section. Every TF-IDF row has Euclidean length 1, so with $p = 2$ the distance between
two articles depends only on the words they share. The Manhattan length of a row is **not** fixed: it is smaller for short
articles, so with $p = 1$ the shortest articles end up close to everything.

### &nbsp;3-7. k-NN issue 1 : normalization
*Normalization or scaling must be done before finding k-nearest neighbors. If not, variables with large measuring units
are over-emphasized* (p. 32). The lecture's table, standardized with the average and standard deviation of p. 32:

In [ ]:
people = pd.DataFrame({'Height': [187, 165, 174, 156, 168], 'Weight': [93, 51, 68, 48, 59], 'BFS': [15, 25, 14, 29, 12]},
                      index=['1', '2', '3', '4', 'N'])
avg = pd.Series({'Height': 165, 'Weight': 65, 'BFS': 20})       # Avg. row of p. 32
std = pd.Series({'Height': 15, 'Weight': 10, 'BFS': 5})         # Stdev. row of p. 32

((people - avg) / std).round(2)

The *[After Normalization]* table of p. 32, number for number. Before it, a difference of 10 cm in height counted ten
times more than a difference of 1 in BFS; after it, every column is measured in standard deviations.

For documents, the variable with the large measuring unit is the **length**. A raw count vector of a long article is
long, and the Euclidean distance mostly measures how many words two articles have. Dividing each row by its length
(`normalize`, as `TfidfVectorizer` does by default) removes that:

In [ ]:
rows = []
for name, A, B in [('raw counts', X_train, X_test),
                   ('counts / length', normalize(X_train), normalize(X_test))]:
    knn = KNeighborsClassifier(n_neighbors=best_k).fit(A, y_train)
    rows.append({'reference data': name, 'test accuracy': knn.score(B, y_test)})
print(pd.DataFrame(rows).to_string(index=False))

dist_raw = euclidean_distances(X_test[q], X_train).ravel()
print('\nraw counts, 5 nearest neighbors of :', test.loc[q, 'title'])
train.loc[np.argsort(dist_raw)[:5], ['title', 'category']].assign(distance=np.sort(dist_raw)[:5].round(2))

**🤔 Quick check** Same words, same $k$, only the scale changes: 0.38 without normalization, 0.82 with it.
Without it, the neighbors of the Argentina article are just other short articles that share none of its words
(penguins, malaria). With it, they were articles about Argentina.

### &nbsp;3-8. k-NN issue 2 : cut-off
*Consider the prior probability of each class* (p. 33): $N(M) = 100$, $N(F) = 400$, and 2 of the 5 neighbors are M.

In [ ]:
neighbors_MF = np.array(['M', 'F', 'M', 'F', 'F'])
p_M = np.mean(neighbors_MF == 'M')
for cutoff in [0.5, 100 / (100 + 400)]:
    print(f'P(X = M) = {p_M}   cut-off {cutoff:.1f}  ->', 'M' if p_M >= cutoff else 'F')

**🤔 Quick check** With the usual cut-off 0.5, X is F. But M is only 20 % of the data, so 2 M neighbors out of 5 is already
twice the base rate: with the cut-off set to the class proportion 0.2, X is M. Our news sample is balanced (25 % each),
so the plain vote is fine there.

## &nbsp;4. Naïve Bayes vs k-NN on the same test articles
Naïve Bayes on the count matrix (`alpha=1`), k-NN on the TF-IDF matrix with the best $k$ of 3-5.

In [ ]:
import time

models = {'Naive Bayes': (MultinomialNB(alpha=1.0), X_train, X_test),
          f'k-NN (k={best_k})': (KNeighborsClassifier(n_neighbors=best_k), T_train, T_test)}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
rows = []
for ax, (name, (model, A, B)) in zip(axes, models.items()):
    t0 = time.time(); model.fit(A, y_train)
    t1 = time.time(); pred = model.predict(B)
    t2 = time.time()
    rows.append({'model': name, 'test accuracy': accuracy_score(y_test, pred),
                 'fit (s)': round(t1 - t0, 4), 'predict (s)': round(t2 - t1, 4)})
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=CLASSES).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()
pd.DataFrame(rows)

**🤔 Quick check** Naïve Bayes is a few points ahead here. The two make the same kind of mistake: Sci/Tech articles
predicted as Business is the largest error of both (technology companies are business news too), and Sports is the
easiest category.
The timing shows how differently they work: Naïve Bayes does its work at training time (counting) and predicts with one
matrix product; k-NN does nothing at training time and compares every test article with all 4,000 training articles.

## &nbsp;5. Classify your own headline
Both classifiers are trained, so they can classify any text. The headline is counted with the training vocabulary
(words outside it are ignored), and Naïve Bayes shows its decision word by word: one row per word, one column per class,
each number is $count \times \log \hat P(w \mid c)$. The bottom row adds them up with $\log \hat P(c)$, exactly the
product of 2-1, in logarithms. The largest sum wins.

> 🔧 **Change and run again.** Replace the headline with your own and run the two cells again. Try a sports headline,
> a business one, a headline that mixes two topics, or *Apple pie recipe wins baking contest*.

In [ ]:
headline = 'Apple unveils a new iPod with a bigger screen'      # 🔧 write your own headline and run again

x = vec.transform([headline])                                    # counts with the training vocabulary
words = vocab[x.indices]
print('words the model knows :', list(words))

table = pd.DataFrame(nb.feature_log_prob_[:, x.indices].T * x.data[:, None], index=words, columns=CLASSES)
total = table.sum() + nb.class_log_prior_                        # log P(c) + sum of the word rows
table.loc['log P(c)'] = nb.class_log_prior_
table.loc['score (sum)'] = total
print('Naive Bayes :', CLASSES[int(np.argmax(total))],
      '  P(c | headline) =', {c: round(float(p), 3) for c, p in zip(CLASSES, nb.predict_proba(x)[0])})
table.round(2)

And k-NN: the same headline as a TF-IDF row, its five nearest training articles, and the vote of the best $k$ of 3-5.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k).fit(T_train, y_train)
t = tfidf.transform([headline])
dist, idx = knn.kneighbors(t, n_neighbors=5)

print(f'k-NN (k={best_k}) :', CLASSES[knn.predict(t)[0]])
train.loc[idx[0], ['title', 'category']].assign(distance=dist[0].round(3))

**🤔 Quick check** For the iPod headline, *apple*, *ipod* and *screen* pull hard toward Sci/Tech, while *new* and *bigger*
hardly matter. Both classifiers agree. The table explains any
other headline the same way: find the words with the largest numbers in the winning column. *Apple pie recipe wins
baking contest* still goes to Sci/Tech: in these 4,000 training articles, *apple* occurs 52 times and always in Sci/Tech.
The classifier does not know what an apple is, only where the word was counted. A headline whose words are all outside
the vocabulary has an empty row: Naïve Bayes then falls back on the prior (0.25 for every class), and k-NN has no
nearest neighbor to speak of.

---
### ✅ What we practiced


| Lecture | In code |
|---|---|
| Definition (p. 8-10) | a document becomes a row of counts; vocabulary from the training set only |
| Naïve Bayes example (p. 17-19) | the products of the two reviews ($120$ vs $0.05$, $0.012$ vs $3.15$, $\times 10^{-8}$) |
| MLE and its limitation (p. 16) | `counts / counts.sum(axis=1)`; more than half of the test articles get probability 0 for every class |
| Laplace smoothing (p. 20) | `(counts + 1) / (counts.sum(axis=1) + V)`, identical to `MultinomialNB(alpha=1.0)` |
| k-NN steps 1-5 (p. 26-31) | Minkowski $p$, majority vs weighted voting, $k = 1$ / $15$ / $151$ on 2-D points, the best $k$ on validation data |
| Normalization, cut-off (p. 32-33) | raw counts vs length-normalized rows; cut-off at the class proportion |
| both classifiers on new text | a headline of your own, explained word by word |

**Next : Notebook 2** puts these two next to the other classifiers of p. 11, and looks at the *matrix-based* representation
of p. 13.